# 05 — Document Classification with `ai_classify`

## Objective

Understand how a Databricks AI Function can assign a document to one of a fixed set of categories, without training a classifier or writing per-category keyword rules. This notebook classifies the 15 documents from Notebook 02 -- which already carry a known, hand-assigned `category` -- so you can measure `ai_classify` against ground truth, not just eyeball its output.

## What We Will Learn

- How `ai_classify` turns free text plus a list of category labels into exactly one of those labels
- Why this is *prompt-based* classification -- there is no training step, no labeled training set, no model to fit -- and what that trades off against a trained classifier
- How the wording of a category label (a bare name vs. a short description) can change classification behavior on ambiguous documents
- How to validate an AI Function's output against ground truth, and against its own contract (did it actually return one of the labels you gave it?)

## Prerequisites

- Completed `02_synthetic_data_generation.ipynb` with matching catalog/schema widget values -- this notebook reads the `documents` Delta table it created
- `ai_classify` is a **preview / entitlement-gated AI Function**, same family as `ai_parse_document` from Notebook 04 -- it requires Unity Catalog, a supported Databricks Runtime, and availability in your workspace's region. If it isn't enabled, the classification query below will fail with a function-not-found or permission error; see **Common Errors / Limitations**
- Optional: `04_document_parsing_ai_parse_document.ipynb`, if you want to feed this notebook's classifier a document you parsed yourself instead of the fixed text used in Step 5

## Conceptual Explanation

**What `ai_classify` does.** It is a Databricks-native AI Function -- callable from SQL or Python, same family as `ai_parse_document` -- that takes a piece of text and an array of candidate labels, and returns exactly one of those labels. Under the hood it's an LLM call, but the output is constrained: you don't get free-form prose back, you get one string, guaranteed to be a member of the label array you passed in. That's the "structured output" the README topic list refers to -- structure isn't bolted on afterward by parsing the model's words, it's a property of what the function can return.

**Prompt-based, not trained.** A traditional text classifier (say, a scikit-learn model) needs a labeled training set, a training step, and produces a model artifact you version and redeploy when categories change. `ai_classify` needs none of that -- you supply the categories at query time, as data. Add a sixth category next quarter and it's a new array element, not a retraining job. The cost of that flexibility: classification quality depends entirely on how well your label *wording* describes the category, since the model has never seen examples of your specific categories, only their names/descriptions and the document text.

**Why label wording matters.** A bare label like `"Operations"` and a descriptive one like `"Operations: an internal operational or branch procedure"` can produce different answers on the same ambiguous document, because the model is reasoning from the label text itself, not from a fixed internal definition of "Operations" your organization uses. Step 4 below deliberately re-runs the same documents with more descriptive labels so you can see whether -- and where -- the answer changes.

**Validation, two ways.** This notebook validates `ai_classify` output in two different senses that are worth keeping distinct:
1. **Against ground truth** -- Notebook 02's documents already have a hand-assigned `category`. Comparing `ai_classify`'s answer to that known-correct label gives you an actual accuracy number on this small dataset.
2. **Against the function's own contract** -- separately from whether the *answer* is right, you can check whether the returned value is even a member of the label set you supplied. This should always be true by construction, but checking it once is a cheap habit worth keeping for any AI Function whose guarantees you haven't personally verified in your workspace.

## Example Data

The 15 documents from `02_synthetic_data_generation.ipynb` (5 categories x 3 documents each: Product, Operations, Compliance, Customer Service, Technical) -- these already carry a `category` column, which this notebook treats as ground truth to measure against.

Additionally, a short piece of text reconstructed from the **Fraud Escalation Playbook** used in Notebook 04 -- a document that was never given a Notebook-02-style category, and that plausibly straddles Operations (it's a procedure) and Compliance (it's about fraud escalation). It's a good stress test for a boundary case where reasonable people could disagree, not just a repeat of the labeled dataset.

## Implementation

### Step 1 — Point at the same catalog/schema as Notebook 02

In [ ]:
dbutils.widgets.text("catalog_name", "main", "Unity Catalog catalog")
dbutils.widgets.text("schema_name", "genai_lab", "Schema")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
documents_table = f"{catalog_name}.{schema_name}.documents"

print(f"Documents table: {documents_table}")

### Step 2 — Load the labeled documents

Same table Notebook 02 wrote. `category` here is the ground truth we hand-assigned -- `ai_classify` never sees this column, only `content`.

In [ ]:
documents_df = spark.table(documents_table)
documents_df.createOrReplaceTempView("documents_v")
display(documents_df.select("doc_id", "title", "category"))

### Step 3 — Classify every document with `ai_classify`, using the bare category names

`ai_classify(content, labels)` takes the text to classify and an array of candidate labels, and returns one of them. We pass exactly the five category names Notebook 02 used, so a "correct" answer here means an exact string match against `actual_category`.

In [ ]:
%sql
SELECT
  doc_id,
  title,
  category AS actual_category,
  ai_classify(
    content,
    ARRAY('Product', 'Operations', 'Compliance', 'Customer Service', 'Technical')
  ) AS predicted_category
FROM documents_v
ORDER BY doc_id

### Step 4 — Capture the result in Python and score it against ground truth

Same query, kept as a DataFrame this time so we can compute accuracy and pull out the specific rows `ai_classify` got wrong -- those mismatches are more informative than the accuracy number itself.

In [ ]:
classified_df = spark.sql(
    """
    SELECT
      doc_id,
      title,
      category AS actual_category,
      ai_classify(
        content,
        ARRAY('Product', 'Operations', 'Compliance', 'Customer Service', 'Technical')
      ) AS predicted_category
    FROM documents_v
    """
).cache()

total = classified_df.count()
correct = classified_df.filter("actual_category = predicted_category").count()
print(f"Accuracy on {total} labeled documents: {correct}/{total} ({100 * correct / total:.1f}%)")

print("\nMismatches (worth reading, not just counting):")
display(classified_df.filter("actual_category != predicted_category"))

### Step 5 — Re-run with descriptive labels instead of bare names

Same documents, same query shape -- only the label *wording* changes, from a bare category name to a one-line description of what belongs in it. If any mismatch from Step 4 disappears here, that's a concrete demonstration of label wording changing model behavior, not the underlying document changing.

In [ ]:
descriptive_labels = [
    "Product: describes a bank account, deposit, or credit product and its terms",
    "Operations: describes an internal operational or branch procedure, such as cash or wire handling",
    "Compliance: describes a regulatory, KYC, AML, or data privacy policy",
    "Customer Service: describes a customer-facing support process, such as disputes or card replacement",
    "Technical: describes an API, authentication, or engineering reference document",
]

labels_sql_array = "ARRAY(" + ", ".join(f"'{label}'" for label in descriptive_labels) + ")"

classified_v2_df = spark.sql(
    f"""
    SELECT
      doc_id,
      title,
      category AS actual_category,
      ai_classify(content, {labels_sql_array}) AS predicted_category_v2
    FROM documents_v
    """
).cache()

comparison_df = classified_df.join(classified_v2_df, on=["doc_id", "title", "actual_category"])
display(
    comparison_df.select(
        "doc_id", "title", "actual_category", "predicted_category", "predicted_category_v2"
    )
)

Note that `predicted_category_v2` will contain the *full descriptive string* you passed in, not the short category name -- because that descriptive string **is** the label. If you want v2's output to compare cleanly against `actual_category`, you'd need to map each descriptive label back to its short name yourself (e.g. a small `CASE WHEN` or Python dict), or extract the leading word before the colon. That mapping step is itself worth noticing: it's manual bookkeeping you take on in exchange for label wording that's easier for the model to reason about.

### Step 6 — Classify an unlabeled, genuinely ambiguous document

The Fraud Escalation Playbook from Notebook 04 was never assigned a Notebook-02-style category. Its text is reconstructed here directly (not re-parsed via `ai_parse_document`), so this step doesn't depend on that notebook's live, workspace-specific output schema -- only on the PDF's known content.

In [ ]:
fraud_escalation_text = (
    "Fraud Escalation Playbook\n\n"
    "This playbook defines how Aurora Trust Bank staff escalate suspected fraud cases. "
    "Any transaction flagged by fraud monitoring must be reviewed within one business hour. "
    "Confirmed fraud cases are escalated to the Fraud Operations team, who freeze the affected "
    "account and notify the customer through the contact center. "
    "Escalation Steps: (1) Analyst reviews the flagged transaction. (2) Analyst confirms or "
    "dismisses the fraud indicator. (3) Confirmed cases are routed to Fraud Operations. "
    "(4) Fraud Operations freezes the account and opens a case file. "
    "Escalation Tiers: Tier 1 handles amounts under 1,000 units; Tier 2 handles 1,000-10,000 "
    "units and requires a supervisor sign-off; Tier 3 handles amounts above 10,000 units and "
    "requires notifying the Compliance department."
)

fraud_df = spark.createDataFrame([("fraud_escalation_playbook", fraud_escalation_text)], ["title", "content"])
fraud_df.createOrReplaceTempView("fraud_doc_v")

display(
    spark.sql(
        """
        SELECT
          title,
          ai_classify(
            content,
            ARRAY('Product', 'Operations', 'Compliance', 'Customer Service', 'Technical')
          ) AS predicted_category
        FROM fraud_doc_v
        """
    )
)

### Step 7 — Validate the function's own contract, not just the answer

Separately from whether the classification is *correct*, confirm the returned value is actually a member of the label array you supplied. `ai_classify` is documented to guarantee this by construction, but this notebook hasn't been run against a live workspace -- checking it once, explicitly, is cheaper than assuming it and finding out otherwise three notebooks from now.

In [ ]:
allowed_categories = ["Product", "Operations", "Compliance", "Customer Service", "Technical"]

invalid_predictions = classified_df.filter(~classified_df.predicted_category.isin(allowed_categories))
invalid_count = invalid_predictions.count()

if invalid_count == 0:
    print(f"All {classified_df.count()} predictions are members of the supplied label set -- contract holds.")
else:
    print(f"{invalid_count} prediction(s) fell outside the supplied label set:")
    display(invalid_predictions)

## Inspect the Output

- What was the accuracy in Step 4? On 15 hand-written, fairly clean documents, a handful of mismatches is a meaningful signal, not noise -- read the specific rows, not just the percentage.
- For any mismatch, does the *document* actually read as ambiguous to you (e.g. the Wire Transfer Procedure mentions sanctions screening -- Operations or Compliance?), or does the misclassification look like a plain model error? Those call for different responses.
- Did Step 5's descriptive labels change any answer from Step 3? If so, which one, and does the new answer look more or less correct to you?
- What did `ai_classify` pick for the Fraud Escalation Playbook in Step 6? Given it mentions both a procedure (Operations-flavored) and a Compliance department sign-off, either answer is defensible -- the interesting part is *which one* the model picked and whether that matches your own intuition.
- Did Step 7 find any invalid prediction? If your workspace ever returns one, that's a materially different (and more serious) finding than a wrong-but-valid label.

## Experimentation Section

1. Add a sixth, deliberately ambiguous document (e.g. one that mixes Compliance and Customer Service content) and re-run Steps 3-4 -- does `ai_classify` pick a single confident answer, or does its choice feel arbitrary?
2. Reduce the label set to three categories by merging `Customer Service` into `Operations` conceptually (pass only four labels, and manually decide which ground-truth rows should now "count" as correct) -- does accuracy go up or down, and why might collapsing categories help or hurt?
3. Try a deliberately bad label set -- e.g. single letters `'A'`, `'B'`, `'C'`, `'D'`, `'E'` instead of real words -- and see whether `ai_classify` still returns sensible groupings. This isolates how much of the function's usefulness comes from the label *wording* itself.
4. Write your own third version of the descriptive labels from Step 5, tuned specifically to fix whatever mismatch you saw in Step 4, and confirm it actually fixes that case without breaking a previously-correct one.
5. If you completed Notebook 04, extract real paragraph text from your workspace's actual `ai_parse_document` output for the fraud PDF, and classify *that* instead of the reconstructed text in Step 6 -- compare the result.

## Common Errors / Limitations

- **`UNRESOLVED_ROUTINE` / function not found** -- `ai_classify` isn't enabled for your workspace, region, or SQL warehouse/cluster type. Same remedy as `ai_parse_document` in Notebook 04: check release notes or ask a workspace admin.
- **Permission or entitlement errors** -- the function exists but your workspace isn't entitled to call it. Same remedy -- check with an admin.
- **Accuracy on 15 documents is not a benchmark** -- this dataset is sized for inspectability, not statistical significance. Don't generalize "X% accuracy" beyond this notebook.
- **Labels are not free** -- like `ai_parse_document`, each `ai_classify` call is a managed AI service invocation. Don't loop it row-by-row over a large table without first confirming cost/throughput expectations; the `SELECT ... FROM documents_v` pattern above already applies it set-at-a-time, which is the right shape to scale from.
- **Category drift over time** -- because labels are supplied at query time rather than baked into a trained model, it's easy to end up with two notebooks or two pipelines using slightly different label wording for "the same" category. Nothing in `ai_classify` itself prevents that; that discipline is on you.
- **Do not assume Step 6's ambiguous-document answer is "the" right answer** -- it demonstrates that ambiguity exists and that the model resolves it one way, not that the resolution is correct. Some documents genuinely belong to more than one category, and a single-label classifier can't express that.

## Summary

You classified 15 known documents with `ai_classify` and measured its accuracy against the ground-truth categories from Notebook 02, then watched how re-wording the labels themselves -- not the documents -- could change the answer on ambiguous cases. You also classified a genuinely ambiguous, unlabeled document (the Fraud Escalation Playbook) and validated that every prediction actually fell within the label set you supplied. The core takeaway: `ai_classify` moves the engineering work from "collect labeled training data and train a model" to "write label text precise enough for a model to reason from" -- a different kind of work, not necessarily less of it.

## Suggested Exercises

- Write down, for each of the five categories, a one-sentence description you'd stand behind as your organization's actual definition -- then compare it against the descriptive labels used in Step 5. Where do they differ, and does that difference matter?
- Pick one mismatch from Step 4 and decide, in your own judgment, whether the *document* is mislabeled in Notebook 02, or the *model* got it wrong. Justify your answer using only the document text.
- When you're ready, move on to **`06_information_extraction_ai_extract.ipynb`**, which goes a step further than a single category label and pulls structured fields (like department or effective date) out of a document's text.